# Cursul 7. Variabile derivate și model
**Fluxuri de lucru pentru știința datelor · Sociologie · 2026**

Analiza exploratorie (C6) a găsit trei tipare: SCI scade cu distanța, este mai mare peste o graniță comună și crește cu migrația. Comunitățile Louvain din C6 au urmat, în mare, granițele țărilor. Caietul continuă povestea cu trei întrebări:

1. **Cât contează fiecare factor, când sunt luați împreună?** Tiparele se suprapun: țările vecine sunt și apropiate, și au mai mulți migranți între ele. Un **model** le include simultan (secțiunile 1-5).
2. **Unde greșește modelul și de ce?** Reziduurile arată ce lipsește; efectele de țară corectează modelul (secțiunile 6-7).
3. **Cum se grupează regiunile?** Întâi după ce sunt: tipurile de regiuni, din indicatori (secțiunea 8). Apoi după cu cine sunt legate: comunitățile din rețeaua „peste model”, comparate cu tipurile, și un model de rețea pe țări (secțiunea 9).

Înainte de model, variabilele se pregătesc (**ingineria variabilelor**, engl. *feature engineering*):

| Etapa | Ce face | Pe datele SCI | Secțiunea |
|---|---|---|---|
| Construcția | variabile noi din cele existente | rata migranților, distanța dintre țări, granița comună | 1 |
| Transformarea | forma distribuției | logaritmul | 2 |
| Scalarea | aceeași scară pentru toate | scorul z | 2 |
| Selecția | setul final, fără redundanță | corelațiile predictorilor, VIF | 3 |
| Extragerea | mai mulți indicatori într-unul | tipurile de regiuni (k-means) | 8 |

Un model pe date observaționale descrie **asocieri**, nu cauze. Fiecare pas are o celulă, iar unde rezultatul se vede pe hartă, o hartă.

## 0. Pregătirea
### 0.1. Bibliotecile
Biblioteca nouă a lecției este **statsmodels**, pentru modele statistice. Se importă în două feluri, cu nume scurte, ca `pd` pentru pandas:
- `smf` (de la *statsmodels formula*): modelele scrise ca formule, `smf.ols("log_sci ~ log_distanta", data=date)`; la stânga lui `~` variabila explicată, la dreapta predictorii;
- `sm`: restul bibliotecii; aici doar `sm.add_constant`, pentru VIF.

In [ ]:
from urllib.request import urlretrieve        # descărcarea fișierelor de pe internet
from sqlalchemy import create_engine          # conexiunea la baza de date (C5)
import numpy as np
import pandas as pd
import geopandas as gpd                       # hărțile (C3, C6)
import seaborn as sns
import matplotlib.pyplot as plt
import networkx as nx                         # rețelele (C6)
import statsmodels.api as sm                  # statsmodels, biblioteca modelelor statistice
import statsmodels.formula.api as smf         # modelele scrise ca formule: smf.ols("y ~ x", data=...)
from statsmodels.stats.outliers_influence import variance_inflation_factor   # VIF
from sklearn.cluster import KMeans            # gruparea k-means
from sklearn.metrics import adjusted_rand_score   # ARI: cât seamănă două împărțiri

sns.set_theme(style="whitegrid")

### 0.2. Datele, din baza de date a cursului
Datele stau în baza de date `dsw.db`, construită în C5. Caietul o descarcă din depozitul cursului (GitHub), deci rulează de oriunde, cu o conexiune la internet. Apoi tabelele se citesc cu SQL, ca în C5: regiunile, perechile de regiuni și perechile de țări.

In [ ]:
DATE = "https://raw.githubusercontent.com/dsw-ubb-2026/-DSW2026/main/curs/data/"   # depozitul cursului

urlretrieve(DATE + "dsw.db", "dsw.db")                  # baza de date din C5, salvată lângă caiet
engine = create_engine("sqlite:///dsw.db")              # conexiunea la baza de date

regiuni = pd.read_sql("select * from regiuni", engine)            # un rând pe regiune
perechi = pd.read_sql("select * from perechi_regiuni", engine)    # un rând pe pereche de regiuni
tari = pd.read_sql("select * from perechi_tari", engine)          # un rând pe pereche de țări

len(regiuni), len(perechi), len(tari)

### 0.3. Harta regiunilor
Contururile regiunilor (GISCO), ca în C3 și C6. Fișierul se descarcă lângă caiet, apoi geopandas îl citește.

In [ ]:
urlretrieve("https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/NUTS_RG_20M_2024_3035_LEVL_2.geojson", "regiuni.geojson")
harta = gpd.read_file("regiuni.geojson")                # un rând pe regiune, cu conturul ei
harta = harta.rename(columns={"NUTS_ID": "regiune"})  # codul regiunii, cu același nume ca în tabelul regiuni
harta = harta[["regiune", "geometry"]]                  # doar codul și conturul
harta = harta.merge(regiuni, on="regiune")              # se adaugă datele regiunilor

### 0.4. Distanța dintre regiuni
Svalbard se exclude (decizia din C6: numitor foarte mic). Distanța dintre două regiuni este distanța dintre centrele lor, ca în C6.

In [ ]:
# Svalbard (NO0B) nu trebuie să apară la niciun capăt al perechii
fara_1 = perechi["regiune_1"] != "NO0B"
fara_2 = perechi["regiune_2"] != "NO0B"
perechi = perechi[fara_1 & fara_2]
len(perechi)

In [ ]:
# centrul fiecărei regiuni; harta are coordonatele în metri
centre = harta.set_index("regiune").centroid

# centrul primei și al celei de-a doua regiuni din fiecare pereche
centru_1 = centre[perechi["regiune_1"]].values
centru_2 = centre[perechi["regiune_2"]].values

# distanța dintre cele două centre, din metri în kilometri (ca în C6)
perechi["distanta_km"] = centru_1.distance(centru_2) / 1000
perechi[["regiune_1", "regiune_2", "distanta_km", "sci"]].head()

## 1. Construcția: variabile noi din cele existente
Modelul se face pe **perechi de țări**, pentru că migrația există doar între țări. Tabelul `tari` are SCI-ul și migranții. Îi lipsesc trei variabile, construite pe rând: rata migranților, distanța dintre țări și granița comună.

### 1.1. Rata migranților
**Cum se citește un rând din `tari`.** Fiecare rând este o **pereche** de țări, nu un drum de la o țară la alta. `tara_1` și `tara_2` sunt cele două țări în ordine alfabetică (IT înaintea lui RO), fără sens de origine sau destinație: SCI-ul este același în ambele sensuri. Migrația are însă două sensuri, păstrate în două coloane construite în C5:

| Coloana | Ce conține | Exemplu, perechea IT-RO |
|---|---|---|
| `din_1_in_2` | cetățenii țării 1 care trăiesc în țara 2 | italienii din România |
| `din_2_in_1` | cetățenii țării 2 care trăiesc în țara 1 | românii din Italia |
| `migranti` | suma celor două sensuri | toți migranții dintre cele două țări |

Modelul folosește suma, pentru că SCI-ul nu are sens: o pereche e legată indiferent în ce direcție au plecat oamenii.

In [ ]:
e_it = tari["tara_1"] == "IT"                     # țara 1 este Italia
e_ro = tari["tara_2"] == "RO"                     # țara 2 este România
it_ro = tari[e_it & e_ro]                         # perechea IT-RO
it_ro[["tara_1", "tara_2", "din_1_in_2", "din_2_in_1", "migranti"]]

Numărul de migranți depinde de mărimea țărilor (C6). Împărțit la populația celor două țări, devine o **rată**, comparabilă între țări mari și mici. Întâi, populația fiecărei țări:

In [ ]:
# datele Băncii Mondiale (C2); NA = Namibia, nu valoare lipsă
bm = pd.read_csv(DATE + "raw/bm.csv", keep_default_na=False, na_values=[""])

populatie = bm[bm["indicator"] == "populatie"]      # doar indicatorul populație
populatie = populatie[populatie["an"] == 2024]      # doar anul 2024
pop = populatie.set_index("tara")["valoare"]        # țara -> populația
pop[["RO", "DE", "LU"]]

Apoi rata: migranții la 1.000 de locuitori ai celor două țări. Se păstrează doar perechile cu migrația cunoscută.

In [ ]:
tari = tari.dropna(subset=["migranti"])           # doar perechile cu migrația cunoscută

pop_1 = tari["tara_1"].map(pop)                   # populația primei țări
pop_2 = tari["tara_2"].map(pop)                   # populația celei de-a doua țări

# migranți la 1.000 de locuitori ai celor două țări
tari["la_mie"] = 1000 * tari["migranti"] / (pop_1 + pop_2)

### 1.2. Numele perechii
**Numele perechii.** `tara_1` și `tara_2` sunt doar cele două țări ale perechii; ordinea lor nu înseamnă nimic (SCI-ul și distanța sunt aceleași în ambele sensuri). Același lucru poate fi însă scris în două feluri: „AT, RO” sau „RO, AT”. Ca fiecare pereche să aibă **un singur nume**, oriunde ar apărea, i se dă un nume scris mereu la fel: cele două coduri în ordine alfabetică, legate cu o liniuță, „AT-RO”. Cu acest nume se leagă apoi toate tabelele: perechile de țări, perechile de regiuni, vecinii.

In [ ]:
def pereche(rand):
    """Numele perechii dintr-un rând: cele două țări, în ordine alfabetică ("RO", "AT" -> "AT-RO")."""
    cele_doua = sorted([rand["tara_1"], rand["tara_2"]])       # ["AT", "RO"]
    return cele_doua[0] + "-" + cele_doua[1]                   # "AT-RO"


tari["pereche"] = tari.apply(pereche, axis=1)          # axis=1: funcția se aplică fiecărui rând
perechi["pereche"] = perechi.apply(pereche, axis=1)    # aceeași funcție, pe perechile de regiuni
tari[["tara_1", "tara_2", "pereche"]].head(3)

### 1.3. Distanța dintre două țări
Distanța dintre două țări se construiește din distanțele dintre regiunile lor: **mediana** distanțelor dintre toate perechile de regiuni ale celor două țări. Mediana nu e trasă de regiunile îndepărtate (insulele, teritoriile de peste mări).

Un exemplu: Austria are 9 regiuni, România 8, deci între ele sunt 72 de perechi de regiuni. Distanța dintre țări este mediana celor 72 de distanțe.

In [ ]:
at_ro = perechi[perechi["pereche"] == "AT-RO"]        # cele 72 de perechi de regiuni
at_ro["distanta_km"].describe().round(0)              # numărul, media, mediana (50%), ...

Același calcul pentru toate perechile de țări, cu `groupby` pe numele perechii; apoi `map` aduce distanța în `tari`, ca populația din 1.1.

In [ ]:
distanta_tari = perechi.groupby("pereche")["distanta_km"].median()   # numele perechii -> distanța
tari["distanta_km"] = tari["pereche"].map(distanta_tari)
tari[["pereche", "distanta_km"]].head()

### 1.4. Granița comună
Două țări sunt vecine dacă se ating. Trei pași: regiunile se lipesc în țări (`dissolve`); fiecare țară se lărgește cu 10 km (`buffer`), ca să acopere micile goluri dintre contururi; se caută țările care se ating (`sjoin`).

In [ ]:
contur = harta[["tara", "geometry"]]                  # țara și conturul fiecărei regiuni
contur = contur.dissolve("tara", as_index=False)      # 1. regiunile aceleiași țări, lipite: o formă pe țară

largit = contur.copy()
largit["geometry"] = contur.buffer(10_000)            # 2. fiecare țară, lărgită cu 10 km (harta e în metri)

atingeri = gpd.sjoin(largit, contur, predicate="intersects")   # 3. un rând pe atingere: tara_left atinge tara_right

In [ ]:
vecini_ro = atingeri[atingeri["tara_left"] == "RO"]["tara_right"]   # pe cine atinge România

ax = contur.plot(color="#eeeeee", edgecolor="white", figsize=(9, 7))
contur[contur["tara"].isin(vecini_ro)].plot(ax=ax, color="#f4a261")   # vecinii (portocaliu)
contur[contur["tara"] == "RO"].plot(ax=ax, color="#2a9d8f")           # România (verde)
ax.set_xlim(2.5e6, 7.4e6)                         # Europa continentală
ax.set_ylim(1.4e6, 5.5e6)
ax.set_title("România și vecinii ei")
ax.set_axis_off()

**Variabila.** Granița este 1 pentru țările vecine, 0 în rest: atingerile primesc numele perechii, iar fiecare pereche din `tari` primește 1 dacă numele ei e printre atingeri (`isin`).

In [ ]:
vecini = atingeri.rename(columns={"tara_left": "tara_1", "tara_right": "tara_2"})   # coloanele cerute de funcție
vecini["pereche"] = vecini.apply(pereche, axis=1)                 # "AT-HU", "AT-DE", ...

se_ating = tari["pereche"].isin(vecini["pereche"])                # True dacă perechea e printre atingeri
tari["granita"] = se_ating.astype(int)                            # 1 sau 0

**Exercițiu.** Câte perechi de țări vecine are tabelul?

In [ ]:
# Suma coloanei granita (0 sau 1): câte perechi au valoarea 1

## 2. Transformarea și scalarea
### 2.1. Logaritmul
SCI, distanța și rata migranților au coadă lungă spre dreapta (C6). Logaritmul le aduce aproape de simetrie și transformă relațiile multiplicative în relații liniare. Granița (0 sau 1) nu se logaritmează: nu are coadă, iar log 0 nu există.

Asimetria înainte și după logaritm:

In [ ]:
variabile = tari[["sci", "distanta_km", "la_mie"]]

inainte = variabile.skew()                   # asimetria pe valorile brute (0 = simetric)
dupa = np.log10(variabile).skew()            # asimetria după logaritm

pd.DataFrame({"înainte": inainte, "după logaritm": dupa}).round(2)

Pe SCI, diferența se vede în histogramă:

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))       # două grafice alăturate

sns.histplot(tari["sci"], ax=ax[0])                   # stânga: SCI, coadă lungă
ax[0].set_title("SCI")

sns.histplot(np.log10(tari["sci"]), ax=ax[1])         # dreapta: log10 SCI, aproape simetric
ax[1].set_title("log10 SCI");

Coloanele noi, pe logaritm:

In [ ]:
# logaritmul în baza 10, ca în C6
tari["log_sci"] = np.log10(tari["sci"])
tari["log_distanta"] = np.log10(tari["distanta_km"])
tari["log_migranti"] = np.log10(tari["migranti"])
tari["log_la_mie"] = np.log10(tari["la_mie"])

In [ ]:
sns.regplot(tari, x="log_distanta", y="log_sci");          # pe log-log, relația e aproape o dreaptă

**Același eșantion pentru toate modelele.** Modelele se compară doar pe exact aceleași perechi. Perechile cu o valoare lipsă se scot o singură dată, aici; tabelul `date` este cel folosit de aici înainte.

In [ ]:
coloane_model = ["log_sci", "log_distanta", "log_la_mie", "granita"]

date = tari.dropna(subset=coloane_model)     # doar perechile cu toate variabilele
date = date.copy()                           # un tabel separat, căruia i se adaugă coloane
len(date)                                    # câte perechi intră în toate modelele

### 2.2. Scalarea: scorul z
Distanța (log) merge de la aproximativ 2 la 3,6, rata migranților (log) de la −3 la 1: scări diferite. **Scorul z** aduce fiecare variabilă la media 0 și abaterea standard 1: z = (x − media) / abaterea standard. Forma distribuției nu se schimbă, doar unitatea. Cu toate variabilele în scor z, coeficienții modelului se pot compara între ei (secțiunea 4.3).

In [ ]:
def scor_z(x):
    """Scorul z: câte abateri standard este fiecare valoare față de medie."""
    return (x - x.mean()) / x.std()

In [ ]:
date["z_log_sci"] = scor_z(date["log_sci"])
date["z_log_distanta"] = scor_z(date["log_distanta"])
date["z_log_la_mie"] = scor_z(date["log_la_mie"])

date[["log_distanta", "z_log_distanta", "log_la_mie", "z_log_la_mie"]].describe().round(2)

**Exercițiu.** Ce schimbă scorul z și ce nu? Media, abaterea standard și asimetria ratei migranților, înainte și după.

In [ ]:
# Tabelul date[["log_la_mie", "z_log_la_mie"]], apoi .agg(["mean", "std", "skew"]) și .round(2)

Media și abaterea standard se schimbă (devin 0 și 1); asimetria rămâne aceeași, aproape 0. Scorul z schimbă **unitatea** (centrul și întinderea), nu **forma**. Forma o schimbă logaritmul (secțiunea 2.1). De aceea, în modelul pe scoruri z (secțiunea 4.3), perechile au aceeași poziție una față de alta; doar coeficienții se citesc în abateri standard.

## 3. Selecția: redundanța predictorilor
Doi predictori care spun aproape același lucru nu se pot separa în model: coeficienții lor devin instabili (**multicoliniaritate**). Prima verificare: corelațiile dintre predictori.

In [ ]:
predictori = ["log_distanta", "log_la_mie", "granita"]     # variabilele explicative ale modelului
date[predictori].corr().round(2)                           # corelația fiecărei perechi de predictori

**Cum se citește.** Diagonala are numai 1 (fiecare variabilă cu ea însăși), iar tabelul e simetric: rămân trei numere, cele de sub diagonală. Fiecare devine o propoziție; semnul spune direcția, mărimea cât de strâns.

| Perechea | r | Propoziția |
|---|---|---|
| distanța și migrația | −0,32 | țările mai depărtate au, în medie, o rată a migranților puțin mai mică (legătură slabă) |
| distanța și granița | −0,63 | țările vecine sunt, de regulă, apropiate (legătura cea mai puternică) |
| migrația și granița | 0,33 | între țările vecine sunt, în medie, ceva mai mulți migranți (legătură slabă) |

La granița 0/1, r compară perechile vecine cu cele nevecine. Întrebarea nu este *dacă* predictorii sunt corelați, ci dacă sunt atât de corelați încât modelul să nu-i mai poată despărți. Regula practică: peste aproximativ 0,8 în valoare absolută. Aici maximul este 0,63, deci predictorii se pot separa.

Corelațiile explică și ce se va vedea în secțiunea 4: singură, distanța „preia” o parte din efectul migrației și al graniței, cu care e legată; când acestea intră în model, coeficientul ei scade.

A doua: **VIF** (factorul de inflație a varianței). Fiecare predictor se explică prin ceilalți; dacă R² e mare, predictorul e redundant. VIF = 1 / (1 − R²). Sub 5: acceptabil; între 5 și 10: atenție; peste 10: unul dintre predictori se scoate.

statsmodels are funcția gata făcută, `variance_inflation_factor`. Ea primește tabelul predictorilor, cu o coloană de 1 pentru constantă (`sm.add_constant`), și poziția coloanei verificate.

In [ ]:
X = sm.add_constant(date[predictori])          # predictorii, plus coloana „const” (numai 1)

vif = {}
for coloana in predictori:
    pozitia = X.columns.get_loc(coloana)                      # a câta coloană din X este
    vif[coloana] = variance_inflation_factor(X.values, pozitia)

pd.Series(vif).round(2)

Toate valorile sunt sub 2. Distanța și granița sunt legate (r = −0,63: țările vecine sunt apropiate), dar nu atât încât să nu se poată separa. Toți trei predictorii rămân.

## 4. Modelul
### 4.1. O dreaptă: tipar și reziduu
Un **model liniar** scrie fiecare valoare ca tipar plus reziduu: log SCI = constantă + pantă × log distanță + reziduu.
- **Constanta** (engl. *intercept*, în statsmodels `Intercept`) este valoarea prezisă a lui log SCI când predictorul este 0. Aici log distanță = 0 înseamnă o distanță de 1 km, care nu există între țări: constanta fixează doar înălțimea dreptei și nu se interpretează.
- **Panta** (coeficientul predictorului) spune cu cât se schimbă log SCI când log distanța crește cu 1, adică atunci când distanța se înmulțește cu 10.
- **Metoda celor mai mici pătrate** alege constanta și panta care fac suma pătratelor reziduurilor cât mai mică. **R²** este partea din variația lui log SCI explicată de model, între 0 și 1.

În formula lui `smf.ols`, la stânga lui `~` stă variabila explicată, la dreapta predictorii.

In [ ]:
m1 = smf.ols("log_sci ~ log_distanta", data=date).fit()
m1.params.round(2)                                         # constanta (Intercept) și panta

**Cum se citesc cele două numere.** Modelul este dreapta log SCI = 6,60 − 0,76 × log distanța.

- **Constanta, 6,60**: log SCI prezis când log distanța este 0, adică la 1 km. O asemenea distanță nu există între țări, deci numărul nu se interpretează; el fixează doar înălțimea dreptei.
- **Panta, −0,76**: când log distanța crește cu 1, adică distanța se **înmulțește cu 10**, log SCI scade cu 0,76. În SCI: 10<sup>−0,76</sup> = 0,17, deci perechea e de aproximativ 6 ori mai puțin legată. Pentru o distanță **dublă**: 2<sup>−0,76</sup> = 0,59, adică cu 41% mai puțin.

Un exemplu: la 1.000 km, log SCI prezis = 6,60 − 0,76 × 3 = 4,32, adică un SCI de aproximativ 21.000. La 2.000 km, aproximativ 12.400.

In [ ]:
print("R² =", round(m1.rsquared, 2))                       # cât explică distanța singură

**R² = 0,16**: distanța singură explică 16% din variația lui log SCI între perechile de țări. Restul de 84% vine din altceva: migrație, graniță și tot ce nu e în model. Secțiunea următoare adaugă primele două.

### 4.2. Trei variabile, fiecare cu o explicație
Un predictor intră în model doar dacă există un motiv teoretic pentru el și o direcție așteptată:

| Variabila | De ce ar conta | Semnul așteptat |
|---|---|---|
| distanța | contactul costă mai mult la distanță mare (modelul gravitațional) | negativ |
| rata migranților | cei plecați păstrează legăturile cu cei de acasă (rețelele de migrație) | pozitiv |
| granița comună | contact zilnic, istorie și minorități comune | pozitiv |

Secțiunea 3 a arătat că cele trei nu se suprapun (VIF sub 2). **M2** le conține pe toate trei și se compară cu M1, distanța singură.

In [ ]:
formula_m2 = "log_sci ~ log_distanta + log_la_mie + granita"
m2 = smf.ols(formula_m2, data=date).fit()

modele = pd.DataFrame({"M1": m1.params, "M2": m2.params})   # câte o coloană pe model
modele.loc["R²"] = [m1.rsquared, m2.rsquared]
modele.round(2)

- Distanța are semn negativ în ambele modele, dar coeficientul ei scade de la −0,76 (M1) la −0,30 (M2): o parte din „efectul distanței” din M1 era migrație, pentru că țările apropiate au mai mulți migranți între ele.
- R² crește de la 0,16 la 0,53: migrația și granița explică mult peste distanță.
- Toate trei au semnul așteptat.

**Exercițiu.** Același model cu **numărul** de migranți (`log_migranti`) în locul ratei. Cât este R²? Ce arată despre construcția variabilelor din secțiunea 1?

In [ ]:
# Formula lui M2, cu log_migranti în locul lui log_la_mie; apoi rsquared pentru ambele modele

Cu numărul de migranți, R² este 0,26; cu rata, 0,53. Aceeași informație, construită altfel, explică de două ori mai mult: acesta este rostul variabilelor derivate.

### 4.3. Care variabilă contează mai mult?
Coeficienții din M2 au unități diferite. Pe scorurile z (secțiunea 2.2), fiecare coeficient spune cu câte abateri standard se schimbă log SCI când predictorul crește cu o abatere standard, deci se pot compara.

In [ ]:
m2z = smf.ols("z_log_sci ~ z_log_distanta + z_log_la_mie + granita", data=date).fit()   # M2, pe scorurile z
m2z.params.round(2)

Rata migranților are coeficientul cel mai mare (0,62), de aproape patru ori mai mare decât distanța (−0,16). Între perechile de țări, migrația contează mai mult decât distanța, odată ce amândouă sunt în model. Granița rămâne pe scara 0/1: coeficientul ei (0,22) este diferența dintre țările vecine și celelalte, în abateri standard ale lui log SCI.

## 5. Rezultatul complet și interpretarea
### 5.1. Ce conține `summary()`
`m2.summary()` dă tot rezultatul modelului, în trei blocuri: modelul în ansamblu (sus), coeficienții (mijloc), diagnosticul reziduurilor (jos).

In [ ]:
m2.summary()

Ce contează din cele trei blocuri (în slide-uri, fiecare câmp are explicația lui):
- **sus**: `R-squared: 0.530`, modelul explică 53% din variația lui log SCI;
- **mijloc**: pentru fiecare predictor, `coef` (estimarea), `P>|t|` (valoarea p, ca în C6) și `[0.025 0.975]` (intervalul de 95%). Migrația: 0,44, p practic 0. Distanța: −0,30, p = 0,004. Granița: 0,11, p = 0,21, intervalul conține 0;
- **jos**: diagnosticul reziduurilor; reziduurile nu sunt perfect normale (`Prob(Omnibus)` sub 0,05), ceea ce se spune în raport.

Erorile standard presupun perechi independente; perechile aceleiași țări nu sunt, iar secțiunea 7 corectează asta.

**Cum se citesc coeficienții pe logaritmi.** Când SCI și predictorul sunt amândouă pe logaritm, coeficientul se citește în **procente** (o *elasticitate*): cu cât la sută se schimbă SCI când predictorul crește cu 1%. Nu contează că distanța e în kilometri și migrația în migranți la mie: totul devine procente.

- **Rata migranților, 0,44**: o rată cu 10% mai mare merge cu un SCI cu aproximativ 4% mai mare, la aceeași distanță și graniță.
- **Distanța, −0,30**: o distanță cu 10% mai mare merge cu un SCI cu aproximativ 3% mai mic, la aceeași migrație și graniță.
- **Granița, 0,11**: nu e pe logaritm (e 0 sau 1), deci nu e o elasticitate; spune doar că, la aceeași distanță și migrație, țările vecine au un SCI puțin mai mare. Intervalul ei conține 0, deci diferența nu se deosebește clar de întâmplare.

### 5.2. Interpretarea, ca într-un raport
Pe 263 de perechi de țări europene, un model liniar cu distanța, rata migranților și granița comună explică 53% din variația lui log SCI (R² ajustat 0,52). **Migrația este predictorul cel mai puternic**: la aceeași distanță și graniță, perechile cu o rată mai mare a migranților sunt mai legate (coeficient 0,44, interval de 95% între 0,38 și 0,51, t = 13,7). **Distanța rămâne semnificativă, dar mult mai slabă decât părea singură**: coeficientul ei scade de la −0,76 în M1 la −0,30 în M2, pentru că o parte din efectul ei trecea prin migrație. **Granița comună nu adaugă un efect clar** odată ce distanța și migrația sunt în model (p = 0,21). Rezultatele sunt asocieri între perechi de țări, nu efecte cauzale; reziduurile nu sunt perfect normale, iar erorile standard sunt subestimate, pentru că aceeași țară apare în multe perechi.

Paragraful descrie M2. Secțiunea 6 arată de ce M2 nu e modelul final, iar secțiunea 7 îl corectează.

## 6. Reziduurile: peste și sub model
**Reziduul** = log SCI observat − log SCI prezis de M2. Pozitiv: perechea e mai legată decât prezic distanța, migrația și granița. Negativ: mai puțin legată. 10 la puterea reziduului spune de câte ori.

In [ ]:
date["prezis"] = m2.fittedvalues                          # log SCI prezis de M2
date["reziduu"] = m2.resid                                 # observat - prezis

In [ ]:
ax = sns.scatterplot(date, x="prezis", y="log_sci")
ax.axline((4, 4), slope=1, color="black")                 # pe linie: exact cât prezice modelul
ax.set(xlabel="log SCI prezis de M2", ylabel="log SCI observat");

**Cum se citește graficul.**
- Fiecare punct este o pereche de țări: pe orizontală, log SCI **prezis** de M2; pe verticală, log SCI **observat**.
- Linia neagră este „predicția perfectă”: punctele de pe ea au exact SCI-ul prezis de model.
- **Distanța pe verticală până la linie este reziduul.** Deasupra liniei: perechea e mai legată decât prezice modelul (reziduu pozitiv). Dedesubt: mai puțin legată (reziduu negativ).
- Norul urmează linia: modelul prinde tiparul general (R² = 0,53). Dar punctele se împrăștie de obicei cu ±0,3-0,5 pe logaritm în jurul ei, adică perechile sunt de 2-3 ori mai legate sau mai puțin legate decât prezice modelul.

Cazurile extreme sunt cele mai interesante:
- **sus, departe de linie**: Luxemburg-Portugalia (prezis 4,8, observat 6,0: de 16 ori peste model) și perechile cu Islanda (de 8-11 ori peste);
- **jos, departe de linie**: perechile cu Turcia (Italia, Ungaria, Cehia, România: de 5-7 ori sub model).

Luxemburgul are o comunitate portugheză foarte mare raportată la populația lui; Islanda e o țară mică (numitor mic, ca Svalbard în C6); Turcia folosește Facebook altfel și are altă limbă. Niciunul dintre acești factori nu este în model: reziduurile arată ce lipsește. Tabelele de mai jos dau perechile extreme.

Ghicitoarea din C1, închisă: partenerii României, față de model.

**Reziduul, tradus.** Reziduul este pe logaritm, deci un număr ca 0,53 nu spune mult singur. Tradus înapoi în SCI, el devine un raport: **SCI observat / SCI prezis**, adică de câte ori este perechea mai legată decât prezice modelul. Peste 1: peste model; sub 1: sub model.

Pașii: perechile cu România (numele perechii conține „RO”), partenerul (numele perechii fără „RO”), SCI-ul prezis și raportul.

In [ ]:
cu_ro = date[date["pereche"].str.contains("RO")]           # perechile cu România: "AT-RO", "RO-SE", ...
cu_ro = cu_ro.copy()                                       # un tabel separat, căruia i se adaugă coloane

cu_ro["partener"] = cu_ro["pereche"].str.replace("RO", "").str.replace("-", "")   # "AT-RO" -> "AT"
cu_ro[["pereche", "partener"]].head(3)

In [ ]:
cu_ro["sci_prezis"] = 10 ** cu_ro["prezis"]               # din log SCI prezis, înapoi în SCI
cu_ro["de_cate_ori"] = cu_ro["sci"] / cu_ro["sci_prezis"]   # observat / prezis

cu_ro = cu_ro.sort_values("de_cate_ori", ascending=False)  # de la cel mai mult peste model
cu_ro[["partener", "sci", "sci_prezis", "de_cate_ori"]].round(1)

**Cum se citește un rând.** Islanda: SCI-ul observat cu România este 34.880; modelul, din distanță, migrație și graniță, prezice aproximativ 10.400. Perechea este deci de 3,4 ori mai legată decât prezice modelul. Italia, la celălalt capăt: observat 69.300, prezis 99.700, deci 0,7: cu aproximativ 30% sub model, deși are cei mai mulți migranți români.

Raportul `de_cate_ori` este exact 10 la puterea reziduului (`10 ** cu_ro["reziduu"]`): reziduul este logaritmul lui.

Apoi harta: fiecare partener, colorat după reziduul lui. Albastru: peste model; roșu: sub model; gri: fără date.

In [ ]:
reziduu_ro = cu_ro.set_index("partener")["reziduu"]           # partener -> reziduu
contur["reziduu"] = contur["tara"].map(reziduu_ro)               # reziduul fiecărui partener (România: gol)
ax = contur.plot(column="reziduu", cmap="RdBu", vmin=-0.5, vmax=0.5, legend=True, figsize=(9, 7),
                 missing_kwds={"color": "#eeeeee"})
ax.set_xlim(2.5e6, 7.4e6)                         # Europa continentală
ax.set_ylim(1.4e6, 5.5e6)
ax.set_title("Partenerii României: peste (albastru) și sub (roșu) model")
ax.set_axis_off()

Sub model sunt mai ales perechile cu Turcia; peste model, perechile cu Islanda. Partenerii României: Islanda, Luxemburg, Irlanda, Ungaria și Austria sunt peste model; Italia este sub model (un SCI cu aproximativ 30% mai mic decât prezice modelul), deși are cei mai mulți migranți români. Pentru Ungaria, explicația probabilă este minoritatea maghiară din România, pe care modelul nu o include. Islanda și Luxemburgul sunt țări mici: un numitor mic, ca Svalbard în C6.

**Exercițiu.** Partenerii Turciei (`"TR"`), de la cel mai mult sub model la cel mai mult peste.

In [ ]:
# 1. Perechile cu Turcia: numele perechii conține "TR" (str.contains), ca pentru România
# 2. Ordonate după reziduu (sort_values); afișate coloanele pereche, sci, la_mie, reziduu

**Reziduul mediu pe țară.** Fiecare pereche are două țări, deci reziduul ei se numără pentru amândouă. Media pe țară arată dacă unele țări sunt sistematic peste sau sub model.

In [ ]:
# reziduul fiecărei perechi, o dată pentru țara 1 și o dată pentru țara 2
pentru_tara_1 = date[["tara_1", "reziduu"]].rename(columns={"tara_1": "tara"})
pentru_tara_2 = date[["tara_2", "reziduu"]].rename(columns={"tara_2": "tara"})
pe_tara = pd.concat([pentru_tara_1, pentru_tara_2])     # un rând pe țară și pereche

media_pe_tara = pe_tara.groupby("tara")["reziduu"].mean()
media_pe_tara.sort_values().round(2)

Reziduurile nu sunt răspândite la întâmplare. Sub model sunt țările mari: Turcia (−0,32), Franța, Spania, Italia. Peste model sunt țările mici: Islanda (0,68), Luxemburgul (0,42), țările baltice, Slovenia. Lipsește deci din model ceva **de nivelul țării**, legat în primul rând de mărimea ei (un SCI calculat pe puțini utilizatori iese mare cu aproape oricine). Secțiunea 7 îl adaugă.

## 7. Modele mai bune: efectele de țară (M3) și M4
Reziduul mediu pe țară (secțiunea 6) a arătat că reziduurile mari nu sunt răspândite la întâmplare: se adună în jurul câtorva țări (Islanda, Luxemburgul, Turcia). Asta înseamnă că lipsește din model ceva **de nivelul țării**: cât de mult se folosește Facebook acolo, cât de mică e țara (numitor mic), cât de izolată e lingvistic.

**Efectele fixe** rezolvă asta fără să măsoare acești factori: fiecare țară primește o variabilă 0/1, egală cu 1 dacă țara apare în pereche. Modelul absoarbe astfel tot ce e constant la nivelul țării, iar coeficienții se citesc **în interiorul țării**: pentru aceeași țară, partenerii mai apropiați, cu mai mulți migranți, sunt mai legați?

### 7.1. Variabilele de țară
Pentru fiecare țară, o coloană 0/1: 1 dacă țara apare în pereche. Se folosește numele perechii, ca la partenerii României: `fe_RO` este 1 pentru „AT-RO”, „RO-SE” etc. O țară se lasă deoparte, ca referință: altfel variabilele de țară ar repeta constanta modelului. La final, numele coloanelor se lipesc într-un text pentru formulă, cu `join`.

In [ ]:
toate = pd.concat([date["tara_1"], date["tara_2"]])    # codurile din ambele coloane
tarile = sorted(toate.unique())                        # cele 24 de țări, o singură dată, alfabetic

coloane_fe = []                                     # numele variabilelor de țară
for t in tarile[1:]:                                # prima țară (AT) rămâne referința
    in_pereche = date["pereche"].str.contains(t)    # țara apare în numele perechii? (ca la România)
    date["fe_" + t] = in_pereche.astype(int)        # 1 dacă da, 0 dacă nu
    coloane_fe.append("fe_" + t)

efecte = " + " + " + ".join(coloane_fe)             # pentru formulă: " + fe_BE + fe_BG + ..."

date[["tara_1", "tara_2", "fe_RO", "fe_IT"]].head()

### 7.2. Modelul M3
Aceeași formulă ca M2, plus variabilele de țară.

In [ ]:
formula_m3 = formula_m2 + efecte                  # formula lui M2, plus variabilele de țară
m3 = smf.ols(formula_m3, data=date).fit()

comparatie = pd.DataFrame({"M2": m2.params, "M3": m3.params})
comparatie.loc[predictori].round(2)               # doar cele trei variabile

In [ ]:
m2.rsquared, m3.rsquared

R² urcă de la 0,53 la 0,92, dar asta nu înseamnă că modelul a „înțeles” mult mai mult: cea mai mare parte vine din variabilele de țară, care descriu țările, nu legăturile. Ce contează sunt coeficienții celor trei variabile: distanța rămâne −0,30, migrația crește de la 0,44 la 0,57, granița de la 0,11 la 0,15.

### 7.3. Prima verificare: VIF
Cu variabilele de țară printre predictori, VIF-ul celor trei variabile (calculat ca în secțiunea 3) crește: distanța ajunge la aproximativ 3,8. Rămâne sub 5: variabilele se pot separa și cu efectele de țară.

### 7.4. A doua verificare: erorile standard grupate
Erorile standard obișnuite presupun observații independente. Perechile nu sunt: Germania apare în 23 de perechi, iar perechile unei țări seamănă între ele. **Erorile grupate** (engl. *clustered standard errors*) țin cont de asta: perechile care au o țară comună sunt tratate ca un grup. Aici grupele sunt pe două dimensiuni: țara 1 și țara 2. Este o aproximare: perechea AT-RO seamănă și cu perechile în care Austria apare a doua. Corecția completă (erorile diadice) dă aceleași concluzii pe aceste date.

In [ ]:
numar = {}                                        # fiecare țară primește un număr: AT 0, BE 1, ...
for i, t in enumerate(tarile):
    numar[t] = i

grupa_1 = date["tara_1"].map(numar)               # grupul țării 1
grupa_2 = date["tara_2"].map(numar)               # grupul țării 2
grupe = np.column_stack([grupa_1, grupa_2])       # două coloane: perechea e în ambele grupuri

grupuri = {"groups": grupe}                       # cum se grupează perechile

# aceleași formule, estimate cu erorile grupate
m2g = smf.ols(formula_m2, data=date).fit(cov_type="cluster", cov_kwds=grupuri)
m3g = smf.ols(formula_m3, data=date).fit(cov_type="cluster", cov_kwds=grupuri)

Statistica t (coeficientul împărțit la eroarea lui) cu erorile grupate. Peste 2 în valoare absolută: efectul se deosebește de întâmplare.

In [ ]:
t_grupat = pd.DataFrame({"t în M2": m2g.tvalues, "t în M3": m3g.tvalues})   # t = coeficientul / eroarea
t_grupat.loc[predictori].round(1)

Cu erorile grupate, în M2 distanța (t = −1,4) și granița (t = 1,0) nu se mai deosebesc de întâmplare; rămâne doar migrația (t = 8,2). În M3, toate trei trec pragul: distanța (−3,8), migrația (21,3), granița (2,9). Efectele de țară preiau o mare parte din asemănarea dintre perechile aceleiași țări, iar estimările devin mai precise. Concluzia lui M2 despre distanță depindea deci de o presupunere greșită (perechi independente); concluzia lui M3 nu depinde de ea.

### 7.5. Partenerii României, după M2 și după M3
Aceeași hartă ca în secțiunea 6, cu reziduurile ambelor modele. În M3, efectul de mărime dispare (Islanda, Luxemburgul nu mai ies peste model); rămân legăturile cu o explicație proprie.

In [ ]:
cu_ro["reziduu_m3"] = m3.resid                 # pandas potrivește rândurile după index: fiecare pereche, reziduul ei
reziduu_m3_ro = cu_ro.set_index("partener")["reziduu_m3"]
contur["reziduu_m3"] = contur["tara"].map(reziduu_m3_ro)

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
contur.plot(column="reziduu", cmap="RdBu", vmin=-0.5, vmax=0.5, ax=ax[0], missing_kwds={"color": "#eeeeee"})
contur.plot(column="reziduu_m3", cmap="RdBu", vmin=-0.5, vmax=0.5, ax=ax[1], missing_kwds={"color": "#eeeeee"})
for a, titlu in zip(ax, ["după M2", "după M3"]):
    a.set_xlim(2.5e6, 7.4e6)
    a.set_ylim(1.4e6, 5.5e6)
    a.set_title("Partenerii României, " + titlu)
    a.set_axis_off()

După M3: Ungaria (de 2,9 ori peste model, minoritatea maghiară) și Irlanda (de 2,1 ori) rămân peste; Italia ajunge exact cât prezice modelul. „Italia sub model” din M2 era un efect al Italiei ca țară mare, nu al perechii cu România.

### 7.6. Modelul M4: reziduurile cele mai mici
M4 pornește de la M3 și face trei schimbări. Fiecare are o justificare teoretică, stabilită înainte de a vedea efectul asupra reziduurilor.

1. **Numărul de migranți în locul ratei migranților la mie.** În M2, rata era mult mai bună decât numărul (exercițiul din secțiunea 4.2): numărul depinde de mărimea țărilor. În M3, mărimea fiecărei țări este deja în efectele de țară. Rămâne ce măsoară numărul: **câți oameni** fac legătura între cele două țări.
2. **Fostul stat comun.** Cehia-Slovacia este de 2,8 ori peste M3. Cehoslovacia s-a despărțit în 1993; Estonia, Letonia și Lituania au făcut parte din URSS până în 1991. Rudele, colegii și limba înțeleasă reciproc nu dispar odată cu statul.
3. **Familia lingvistică.** Limbile înrudite (romanice, germanice, slave, baltice, uralice) se învață și se înțeleg mai ușor; prieteniile trec mai ușor peste o graniță lingvistică mică.

**Ce nu poate intra.** Orice variabilă a unei singure țări (PIB-ul pe locuitor, șomajul, vârsta medie, folosirea rețelelor sociale) este deja în efectele de țară: pentru Germania, ea are aceeași valoare în toate perechile Germaniei. În M4 pot intra doar variabile ale **perechii**: distanța, migrația dintre cele două țări, granița, istoria și limba comune.

Primele două variabile noi sunt 0/1, ca granița:

In [ ]:
fost_stat = ["CZ-SK", "EE-LT", "EE-LV", "LT-LV"]                # Cehoslovacia; țările baltice (fosta URSS)
date["stat_comun"] = date["pereche"].isin(fost_stat).astype(int)   # 1 pentru aceste perechi, 0 în rest

date[date["stat_comun"] == 1][["pereche", "stat_comun"]]

Familia lingvistică: fiecare țară primește familia limbii majoritare (Belgia: neerlandeza, limba majorității; Irlanda: engleza). Perechea primește 1 dacă cele două țări au aceeași familie.

In [ ]:
familie = {
    "ES": "romanică", "FR": "romanică", "IT": "romanică", "PT": "romanică", "RO": "romanică",
    "AT": "germanică", "BE": "germanică", "DE": "germanică", "IE": "germanică", "IS": "germanică",
    "LU": "germanică", "NL": "germanică", "NO": "germanică", "SE": "germanică",
    "BG": "slavă", "CZ": "slavă", "SI": "slavă", "SK": "slavă",
    "LT": "baltică", "LV": "baltică",
    "EE": "uralică", "FI": "uralică", "HU": "uralică",
    "TR": "turcică",
}

familie_1 = date["tara_1"].map(familie)                  # familia limbii țării 1
familie_2 = date["tara_2"].map(familie)                  # familia limbii țării 2
date["familie_limba"] = (familie_1 == familie_2).astype(int)   # 1 dacă e aceeași familie

date["familie_limba"].sum()                              # câte perechi au limbi înrudite

**M4**: distanța, numărul de migranți, granița, fostul stat comun, familia lingvistică și efectele de țară. Se estimează direct cu erorile grupate (secțiunea 7.4).

In [ ]:
predictori_m4 = ["log_distanta", "log_migranti", "granita", "stat_comun", "familie_limba"]
formula_m4 = "log_sci ~ " + " + ".join(predictori_m4) + efecte    # cele cinci variabile, plus efectele de țară

m4 = smf.ols(formula_m4, data=date).fit(cov_type="cluster", cov_kwds=grupuri)

rezultat = pd.DataFrame({"coeficient": m4.params, "t grupat": m4.tvalues})
rezultat.loc[predictori_m4].round(2)

Toate cinci au semnul așteptat și |t| peste 2. Cele două țări cu limbi înrudite au un SCI de aproximativ 1,15 ori mai mare (10 la puterea 0,06), la aceeași distanță, migrație și istorie. Înainte de a compara reziduurile, aceeași verificare ca pentru M3: VIF, cu efectele de țară printre predictori.

In [ ]:
X4 = sm.add_constant(date[predictori_m4 + coloane_fe])   # predictorii lui M4, plus constanta

vif_m4 = {}
for coloana in predictori_m4:
    pozitia = X4.columns.get_loc(coloana)                 # a câta coloană este
    vif_m4[coloana] = variance_inflation_factor(X4.values, pozitia)

pd.Series(vif_m4).round(1)

Toate sub 5: cele cinci variabile se pot separa.

**Cât de mici sunt reziduurile?** Abaterea standard a reziduurilor este pe logaritm; 10 la puterea ei spune **de câte ori greșește de obicei** modelul (în plus sau în minus).

In [ ]:
abatere = pd.Series({"M2": m2.resid.std(), "M3": m3.resid.std(), "M4": m4.resid.std()})   # pe log10
de_obicei = 10 ** abatere                       # de câte ori greșește de obicei fiecare model

r2 = pd.Series({"M2": m2.rsquared, "M3": m3.rsquared, "M4": m4.rsquared})
pd.DataFrame({"R²": r2, "greșește de obicei de ×": de_obicei}).round(2)

M2 greșește de obicei de 2,2 ori, M4 de 1,32 ori. Cea mai mare parte a câștigului vine din efectele de țară (M3); M4 îl rafinează. Graficul observat față de prezis arată diferența: cu cât punctele sunt mai aproape de linia de 45°, cu atât reziduurile sunt mai mici.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
ax[0].scatter(m2.fittedvalues, date["log_sci"], s=12)   # M2: prezis (orizontal), observat (vertical)
ax[1].scatter(m4.fittedvalues, date["log_sci"], s=12)   # M4

for a, titlu in zip(ax, ["M2", "M4"]):
    a.plot([3, 6], [3, 6], color="gray")                # linia de 45°: observat = prezis
    a.set_title(titlu)
    a.set_xlabel("log SCI prezis")
    a.set_ylabel("log SCI observat")

Perechile care rămân cele mai departe de M4:

In [ ]:
date["reziduu_m4"] = m4.resid                        # observat - prezis, pe log10
date["de_cate_ori_m4"] = 10 ** date["reziduu_m4"]    # observat / prezis

extreme = date.sort_values("de_cate_ori_m4")[["pereche", "de_cate_ori_m4"]]
pd.concat([extreme.head(3), extreme.tail(3)]).round(2)   # cele mai mult sub și cele mai mult peste

Cehia-Slovacia a coborât de la ×2,8 la ×1,24. Rămân peste model Luxemburg-Portugalia (×3,4) și Ungaria-România (×3,1). Ar fi ușor să adăugăm o variabilă „minoritate” pentru Ungaria-România: reziduul ar scădea, dar variabila ar fi aleasă **după** ce am văzut reziduul, deci ar învăța datele pe de rost. O variabilă intră în model pentru că are un motiv teoretic stabilit dinainte, nu pentru că micșorează un reziduu anume.

**Alte variabile ale perechii, verificate.** Cu aceleași date se pot construi și alte variabile ale perechii; niciuna nu aduce ceva în plus:
- **apropierea culturală** (aceeași zonă pe harta culturală World Values Survey: protestantă, catolică, ortodoxă, baltică): t ≈ 0,8; perechile apropiate cultural sunt deja vecine sau cu mulți migranți;
- **diferențele dintre țări** (PIB pe locuitor, vârsta medie, studiile superioare): t sub 1,5; diferența de șomaj trece abia pragul, cu un efect foarte mic.

Harta partenerilor României după M4, ca în secțiunea 7.5:

In [ ]:
cu_ro["reziduu_m4"] = m4.resid                       # reziduul lui M4, pe aceleași perechi
reziduu_m4_ro = cu_ro.set_index("partener")["reziduu_m4"]
contur["reziduu_m4"] = contur["tara"].map(reziduu_m4_ro)

fig, ax = plt.subplots(figsize=(7, 6))
contur.plot(column="reziduu_m4", cmap="RdBu", vmin=-0.5, vmax=0.5, ax=ax, legend=True, missing_kwds={"color": "#eeeeee"})
ax.set_xlim(2.5e6, 7.4e6)
ax.set_ylim(1.4e6, 5.5e6)
ax.set_title("Partenerii României, după M4")
ax.set_axis_off()

**Exercițiu.** Avionul: adăugați la M4 numărul de pasageri ai zborurilor dintre cele două țări (`pasageri`, pe logaritm). Zece perechi nu au zboruri directe și ies din model. Ce arată coeficientul? Poate fi zborul o cauză a prieteniilor?

In [ ]:
# 1. cu_zboruri: perechile cu pasageri > 0; coloana log_pasageri
# 2. m5: formula_m4 + " + log_pasageri", pe cu_zboruri
# 3. Coeficientul și t-ul lui log_pasageri

Coeficientul este pozitiv și se deosebește de întâmplare, dar zborurile nu sunt o cauză: liniile aeriene se deschid acolo unde există deja migranți, rude și prieteni. O asemenea variabilă este mai degrabă o **consecință** a legăturilor; pusă printre predictori, explică SCI-ul prin el însuși. M4 rămâne fără ea.

## 8. Extragerea: tipurile de regiuni (clustering)
**Extragerea** rezumă mai mulți indicatori într-o singură variabilă. Analiza componentelor principale (PCA) dă un scor numeric; **gruparea** (engl. *cluster analysis*) dă o categorie: regiunile asemănătoare pe toți indicatorii ajung în aceeași grupă. Rezultatul este o variabilă derivată nouă, „tipul de regiune”, utilă unui proiect pe regiuni. Gruparea se face pe **regiuni**, nu pe perechi.

**k-means** împarte regiunile în k grupe: fiecare regiune merge la cel mai apropiat centru, apoi fiecare centru se mută în media regiunilor lui, până nu se mai schimbă nimic.

### 8.1. Indicatorii, pe aceeași scară
Cinci indicatori ai regiunilor. Densitatea are coadă lungă, deci intră pe logaritm (secțiunea 2.1).

In [ ]:
reg = regiuni.copy()                                   # o copie a tabelului regiunilor
reg["log_densitate"] = np.log10(reg["densitate"])      # densitatea, pe logaritm

indicatori = ["pib_locuitor", "studii_superioare", "somaj", "varsta_medie", "log_densitate"]
reg = reg.dropna(subset=indicatori)                    # doar regiunile cu toți cei cinci indicatori
len(reg)

Distanța dintre regiuni se calculează pe toți indicatorii deodată, deci indicatorii trebuie să aibă aceeași scară: **scorul z este obligatoriu**. Altfel, PIB-ul (zeci de mii de euro) ar decide singur grupele.

In [ ]:
z = scor_z(reg[indicatori])                                 # scorul z din secțiunea 2.2, pe fiecare coloană
z.describe().round(2)                                       # media 0, abaterea standard 1, la toți

### 8.2. Câte grupe? Graficul cotului
Ca la graficul scree din analiza componentelor principale (PCA), se caută numărul de grupe după care un pas în plus aduce puțin. La k-means, pe axa verticală stă **suma pătratelor din interiorul grupelor** (în scikit-learn se numește `inertia_`): cât de departe sunt regiunile de centrul grupei lor, adunat pe toate regiunile.

- Cu o singură grupă, regiunile sunt departe de centru: suma e mare.
- Fiecare grupă în plus apropie regiunile de un centru, deci suma scade mereu (cu 266 de grupe ar fi 0).
- Se alege k în **cot**: punctul după care curba coboară mult mai încet.

In [ ]:
suma_patrate = []                                      # câte o valoare pentru k = 1, 2, ..., 8 grupe
for k in range(1, 9):
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=1)   # algoritmul, cu k grupe
    grupare = kmeans.fit(z)                                    # aplicat pe scorurile z
    suma_patrate.append(grupare.inertia_)                      # suma pătratelor din interiorul grupelor

plt.plot(range(1, 9), suma_patrate, marker="o")
plt.xlabel("numărul de grupe, k")
plt.ylabel("suma pătratelor în grupe");

### 8.3. Patru grupe
`random_state=1` face rezultatul același la fiecare rulare; `n_init=10` repornește algoritmul de 10 ori și păstrează varianta cea mai bună.

In [ ]:
kmeans = KMeans(n_clusters=4, n_init=10, random_state=1)    # algoritmul, cu 4 grupe
grupare = kmeans.fit(z)                                     # aplicat pe scorurile z
reg["grupa"] = grupare.labels_                              # grupa fiecărei regiuni
reg["grupa"].value_counts()                                 # câte regiuni are fiecare grupă

Profilul fiecărei grupe: mediana indicatorilor, pe scara inițială. După profil se dă un nume fiecărei grupe.

In [ ]:
# indicatorii pe scara inițială (densitatea fără logaritm)
initiali = ["pib_locuitor", "studii_superioare", "somaj", "varsta_medie", "densitate"]

profil = reg.groupby("grupa")[initiali].median()       # mediana fiecărui indicator, pe grupă
profil.round(1)

In [ ]:
grupa_regiunii = reg.set_index("regiune")["grupa"]          # regiunea -> grupa
harta["grupa"] = harta["regiune"].map(grupa_regiunii)       # grupa fiecărei regiuni, pe hartă

ax = harta.plot(column="grupa", categorical=True, cmap="Set2", legend=True, figsize=(9, 7),
                missing_kwds={"color": "#eeeeee"})
ax.set_xlim(2.5e6, 7.4e6)                         # Europa continentală
ax.set_ylim(1.4e6, 5.5e6)
ax.set_title("Patru tipuri de regiuni (k-means pe cinci indicatori)")
ax.set_axis_off()

Curba nu are un cot foarte clar: k = 4 este o alegere, justificată prin grupe care se pot descrie. Numerele grupelor sunt doar etichete și pot ieși în altă ordine; grupele se numesc după profil. Din 298 de regiuni intră 266: celelalte nu au toți indicatorii (Elveția, Norvegia, Balcanii de Vest).

| Tipul de regiune | Regiuni | PIB/loc. | Studii superioare (%) | Șomaj (%) | Vârsta medie | Densitate | Exemple |
|---|---|---|---|---|---|---|---|
| metropolitane | 36 | 60.100 | 50,0 | 4,7 | 41,4 | 740 | capitalele: Praga, Bratislava, Budapesta, Varșovia, București-Ilfov, Madrid, Stockholm |
| dezvoltate | 119 | 38.300 | 34,6 | 3,7 | 45,1 | 139 | Germania, Austria, nordul Italiei, Polonia vestică |
| periferice, îmbătrânite | 90 | 27.350 | 27,9 | 7,7 | 45,8 | 65 | Grecia, sudul Spaniei și al Italiei, Franța rurală, România fără București, Bulgaria |
| tinere, cu șomaj ridicat | 21 | 20.500 | 22,8 | 10,3 | 34,2 | 140 | Turcia, teritoriile franceze de peste mări, Ceuta |

România se împarte: București-Ilfov este în grupa metropolelor, celelalte șapte regiuni în grupa periferiilor. Gruparea nu știe nimic despre țări sau hartă, dar capitalele ies împreună. Ca la comunități, grupele arată *că* regiunile se aseamănă, nu *de ce*.

## 9. Rețeaua: ce rămâne după model
Trei pași: rețeaua țărilor după M3 (perechile mult peste model), un model de rețea (QAP) și comunitățile pe regiuni.

### 9.1. Rețeaua țărilor după M3
O pereche de țări devine o legătură dacă e de cel puțin 1,5 ori mai legată decât prezice M3. Rețeaua folosește M3, nu M4: M3 nu conține fostul stat comun, iar rețeaua arată de unde a pornit această variabilă.

In [ ]:
date["reziduu_m3"] = m3.resid                                  # reziduul fiecărei perechi în M3
legate = date[10 ** date["reziduu_m3"] >= 1.5]                  # perechile de cel puțin 1,5 ori peste M3
legate[["pereche", "reziduu_m3"]].sort_values("reziduu_m3", ascending=False).head(8)

In [ ]:
R = nx.from_pandas_edgelist(legate, "tara_1", "tara_2")          # o legătură pe pereche, ca în C6

harta["x"] = harta.centroid.x                                    # poziția fiecărei țări: mediana centrelor regiunilor
harta["y"] = harta.centroid.y
pozitie = harta.groupby("tara")[["x", "y"]].median()
pozitii = {}
for tara in pozitie.index:
    pozitii[tara] = (pozitie.loc[tara, "x"], pozitie.loc[tara, "y"])

ax = contur.plot(color="#eeeeee", edgecolor="white", figsize=(9, 7))
nx.draw(R, pozitii, ax=ax, with_labels=True, node_size=250, node_color="#2a9d8f", edge_color="#555555", font_size=8)
ax.set_xlim(2.5e6, 7.4e6)
ax.set_ylim(1.4e6, 5.5e6)
ax.set_title("Perechile de țări de cel puțin 1,5 ori peste M3")

Legăturile rămase au fiecare o explicație de verificat: Luxemburg-Portugalia (migrația portugheză), Ungaria-România (minoritatea maghiară), Cehia-Slovacia (fostul stat comun), țările baltice între ele.

### 9.2. Un model de rețea: regresia QAP
Perechile de țări formează și ele o rețea: 24 de noduri, câte o legătură între fiecare două țări. Modelele din secțiunile 4-7 tratează perechile ca rânduri ale unui tabel; în rețea, perechile aceleiași țări nu sunt independente (secțiunea 7.4). **Regresia QAP** (engl. *quadratic assignment procedure*, Krackhardt, 1988) este modelul clasic de rețea pentru astfel de date: aceeași regresie, dar semnificația se calculează prin **permutarea nodurilor**.

Ideea, ca la amestecarea din C6: se amestecă etichetele țărilor (Austria devine, de exemplu, Portugalia), iar fiecare pereche primește SCI-ul perechii „amestecate”. Structura rețelei rămâne aceeași (fiecare țară își păstrează toate legăturile, doar sub alt nume), dar legătura dintre SCI și predictori se rupe. Se repetă de 1.000 de ori: dacă coeficientul real e mai mare decât aproape toți coeficienții „amestecați”, asocierea nu ține de structura rețelei.

In [ ]:
# log SCI al fiecărei perechi de țări, căutat după cele două coduri, în ambele ordini
log_sci_pereche = {}
for a, b, valoare in zip(date["tara_1"], date["tara_2"], date["log_sci"]):
    log_sci_pereche[(a, b)] = valoare
    log_sci_pereche[(b, a)] = valoare

In [ ]:
generator = np.random.default_rng(1)              # amestecări care se pot repeta
coeficienti_amestecati = []

for i in range(1000):
    noile_nume = generator.permutation(tarile)              # țările, în altă ordine
    amestec = dict(zip(tarile, noile_nume))                 # fiecare țară primește numele alteia

    y_nou = []
    for a, b in zip(date["tara_1"], date["tara_2"]):
        pereche_noua = (amestec[a], amestec[b])
        y_nou.append(log_sci_pereche.get(pereche_noua))     # SCI-ul perechii amestecate (lipsă: None)

    copie = date[predictori].copy()
    copie["log_sci"] = y_nou
    copie = copie.dropna()                                  # perechile amestecate fără SCI ies

    model = smf.ols(formula_m2, data=copie).fit()
    coeficienti_amestecati.append(model.params)

amestecati = pd.DataFrame(coeficienti_amestecati)          # un rând pe amestecare

Valoarea p a QAP: proporția amestecărilor în care coeficientul iese, în valoare absolută, cel puțin cât cel real.

In [ ]:
p_qap = {}
for v in predictori:
    real = abs(m2.params[v])                        # coeficientul din M2
    amestecat = amestecati[v].abs()                 # cei 1.000 de coeficienți amestecați
    p_qap[v] = (amestecat >= real).mean()           # proporția celor cel puțin la fel de mari

pd.Series(p_qap)

Migrația trece testul (p sub 0,001): niciuna dintre cele 1.000 de amestecări nu dă un coeficient atât de mare. Distanța (p în jur de 0,2) și granița (p în jur de 0,45) nu trec. Este aceeași concluzie ca la erorile grupate pentru M2 (secțiunea 7.4), obținută pe altă cale: când se ține cont că perechile aceleiași țări seamănă între ele, în M2 rezistă doar migrația. Efectele de țară (M3, M4) rămân necesare.

### 9.3. Comunitățile pe regiuni: modelul de bază
În C6, comunitățile Louvain pe SCI urmau, în mare, țările. Ce comunități rămân după ce distanța și granițele sunt scoase? Întâi, un model de bază pe perechile de regiuni.

In [ ]:
perechi["log_sci"] = np.log10(perechi["sci"])
perechi["log_distanta"] = np.log10(perechi["distanta_km"])

aceeasi = perechi["tara_1"] == perechi["tara_2"]        # cele două regiuni sunt în aceeași țară?
perechi["aceeasi_tara"] = aceeasi.astype(int)           # 1 dacă da, 0 dacă nu

In [ ]:
m_regiuni = smf.ols("log_sci ~ log_distanta + aceeasi_tara", data=perechi).fit()
print("R² =", round(m_regiuni.rsquared, 2))
m_regiuni.params.round(2)

Pe 44 de mii de perechi de regiuni, distanța și granița explică aproape jumătate din variația lui log SCI. Două regiuni din aceeași țară au un SCI de aproximativ 22 de ori mai mare (10 la puterea 1,34) decât două regiuni din țări diferite, la aceeași distanță.

**Greutatea legăturii.** Louvain are nevoie de greutăți pozitive, deci nu se folosește reziduul (care poate fi negativ), ci 10 la puterea lui: **de câte ori** este perechea mai legată decât prezice modelul. 1 = exact cât prezice; 3 = de trei ori mai legată; 0,5 = de două ori mai puțin.

In [ ]:
perechi["peste_model"] = 10 ** m_regiuni.resid          # observat / prezis
perechi["peste_model"].describe().round(2)

**Două rețele, două împărțiri.**
Aceleași regiuni și aceleași perechi, cu două greutăți: SCI-ul (ca în C6) și „peste model”. Louvain dă, pentru fiecare regiune, numărul comunității ei; funcția de mai jos face din rezultat un tabel regiune -> comunitate (ca în C6).

In [ ]:
G_sci = nx.from_pandas_edgelist(perechi, "regiune_1", "regiune_2", edge_attr="sci")
G_peste = nx.from_pandas_edgelist(perechi, "regiune_1", "regiune_2", edge_attr="peste_model")
G_sci.number_of_nodes(), G_sci.number_of_edges()

In [ ]:
def louvain(G, greutate, rezolutie=1, seed=1):
    """Comunitățile Louvain, ca tabel: regiunea -> numărul comunității ei."""
    grupuri = nx.community.louvain_communities(G, weight=greutate, resolution=rezolutie, seed=seed)

    comunitate = {}
    for numar, grup in enumerate(grupuri):        # fiecare grup primește un număr
        for regiune in grup:
            comunitate[regiune] = numar

    tabel = pd.Series(comunitate)
    return tabel.sort_index()                     # regiunile mereu în aceeași ordine

**ARI** (indicele Rand ajustat, engl. *adjusted Rand index*) compară două împărțiri ale acelorași elemente. Se uită la toate perechile de regiuni: sunt împreună în ambele împărțiri, sau separate în ambele? 1 = împărțiri identice; 0 = potrivire cât la întâmplare. Numerele grupelor nu contează, doar cine e împreună cu cine.

Prima comparație: comunitățile față de țări. Cât de mult urmează comunitățile granițele?

In [ ]:
tara_regiunii = harta.set_index("regiune")["tara"]       # regiunea -> țara

comunitati_sci = louvain(G_sci, "sci")                   # pe SCI, ca în C6
comunitati_peste = louvain(G_peste, "peste_model")       # după modelul de bază

tara_sci = tara_regiunii[comunitati_sci.index]           # țara fiecărei regiuni, în aceeași ordine
tara_peste = tara_regiunii[comunitati_peste.index]

# ARI cu țara: cât de mult urmează comunitățile granițele
ari_sci = adjusted_rand_score(tara_sci, comunitati_sci)
ari_peste = adjusted_rand_score(tara_peste, comunitati_peste)

pd.DataFrame({"comunități": [comunitati_sci.nunique(), comunitati_peste.nunique()],
              "ARI cu țara": [ari_sci, ari_peste]}, index=["pe SCI", "după model"]).round(2)

Pe SCI, Louvain găsește aproximativ 15 comunități, iar ARI cu țara este în jur de 0,76: comunitățile sunt, în mare, țările (rezultatul din C6). După model, comunitățile sunt mai puține și mai mari, iar ARI cu țara scade la aproximativ 0,25: comunitățile trec peste granițe.

### 9.4. Alegerea parametrului: rezoluția
Louvain are doi parametri care schimbă rezultatul:
- **rezoluția**: sub 1, comunități mai puține și mai mari; peste 1, mai multe și mai mici;
- **seed**: algoritmul parcurge regiunile într-o ordine aleatoare, deci două rulări pot da împărțiri puțin diferite.

O rezoluție bună dă comunități **stabile**: aproape aceleași la fiecare rulare. Stabilitatea se măsoară tot cu ARI: prima rulare comparată cu alte patru. Celula rulează 25 de împărțiri, în aproximativ un minut.

In [ ]:
rezultate = []
for rezolutie in [0.5, 0.75, 1, 1.25, 1.5]:
    prima = louvain(G_peste, "peste_model", rezolutie, seed=0)        # prima rulare

    acorduri = []
    for seed in [1, 2, 3, 4]:                                          # încă patru rulări
        alta = louvain(G_peste, "peste_model", rezolutie, seed=seed)
        acorduri.append(adjusted_rand_score(prima, alta))              # cât seamănă cu prima

    rand = {
        "rezoluția": rezolutie,
        "comunități": prima.nunique(),
        "stabilitatea (ARI mediu)": np.mean(acorduri),
        "ARI cu țara": adjusted_rand_score(tara_regiunii[prima.index], prima),
    }
    rezultate.append(rand)

pd.DataFrame(rezultate).round(2)

- La rezoluția 0,5, toate regiunile intră într-o singură comunitate: stabilă, dar fără informație.
- Rezoluția 1 dă cinci comunități și stabilitatea cea mai mare (ARI mediu în jur de 0,93): aproape aceleași comunități la fiecare rulare.
- Peste 1, comunitățile se înmulțesc și se sparg diferit de la o rulare la alta, iar ARI cu țara crește: comunitățile mici se apropie din nou de țări.

Se alege rezoluția 1. Numerele exacte pot varia puțin de la un calculator la altul; ordinea lor nu se schimbă.

### 9.5. Cinci comunități peste granițe

In [ ]:
comunitati = louvain(G_peste, "peste_model", rezolutie=1)
harta["comunitate"] = harta["regiune"].map(comunitati)   # comunitatea fiecărei regiuni, pe hartă

In [ ]:
ax = harta.plot(column="comunitate", categorical=True, cmap="Set2", legend=True, figsize=(9, 7),
                missing_kwds={"color": "#eeeeee"})
ax.set_xlim(2.5e6, 7.4e6)                         # Europa continentală
ax.set_ylim(1.4e6, 5.5e6)
ax.set_title("Comunitățile după model: legături peste distanță și granițe")
ax.set_axis_off()

**Aceleași comunități, ca rețea.** Cu 44 de mii de legături, desenul complet ar fi o pată neagră. Se desenează un **schelet**: arborele care leagă toate regiunile prin legăturile cele mai puternice (`maximum_spanning_tree`), plus cele patru legături cele mai puternice ale fiecărei regiuni. Așezarea (`spring_layout`) trage regiunile legate una spre alta. Stânga: rețeaua pe SCI, ca în C6; dreapta: rețeaua „peste model”.

In [ ]:
def schelet(G, greutate, k=4):
    """Pentru desen: arborele legăturilor celor mai puternice, plus cele k legături cele mai puternice ale fiecărei regiuni."""
    S = nx.maximum_spanning_tree(G, weight=greutate)            # leagă toate regiunile
    for regiune in G.nodes:
        vecini = G[regiune]                                     # legăturile regiunii
        greutati = {}
        for v in vecini:
            greutati[v] = vecini[v][greutate]                   # greutatea fiecărei legături
        cele_mai_tari = pd.Series(greutati).nlargest(k).index   # cei k vecini cei mai legați
        for v in cele_mai_tari:
            S.add_edge(regiune, v)
    return S

In [ ]:
S_sci = schelet(G_sci, "sci")
S_peste = schelet(G_peste, "peste_model")

fig, ax = plt.subplots(1, 2, figsize=(16, 8))

pozitii = nx.spring_layout(S_sci, seed=1)                       # regiunile legate stau aproape
culori = [comunitati_sci[r] for r in S_sci.nodes]               # culoarea: comunitatea pe SCI
nx.draw(S_sci, pozitii, ax=ax[0], node_color=culori, cmap="tab20", node_size=20, width=0.2, edge_color="#bbbbbb")
ax[0].set_title("Pe SCI (C6): insule care sunt țările")

pozitii = nx.spring_layout(S_peste, seed=1)
culori = [comunitati[r] for r in S_peste.nodes]                 # culoarea: comunitatea după model
nx.draw(S_peste, pozitii, ax=ax[1], node_color=culori, cmap="Set2", node_size=20, width=0.2, edge_color="#bbbbbb")
ax[1].set_title("Peste model: comunități care trec granițele")

Pe SCI, rețeaua se rupe în ciorchini strânși, aproape fiecare o țară: în interiorul țării, legăturile sunt de zeci de ori mai puternice. După model, ciorchinii dispar, iar regiunile din țări diferite se amestecă în cele cinci comunități.

Compoziția fiecărei comunități: câte regiuni are din fiecare țară (primele șase țări).

In [ ]:
for numar in sorted(comunitati.unique()):
    regiunile = comunitati[comunitati == numar].index         # regiunile comunității
    tarile_ei = tara_regiunii[regiunile].value_counts()       # câte regiuni din fiecare țară
    print(numar, tarile_ei.head(6).to_dict())

Numerele comunităților sunt doar etichete; comunitățile se citesc după compoziție. Pe datele cursului ies cinci blocuri, fiecare cu o explicație plauzibilă, de verificat cu alte date:

| Comunitatea | Regiunile principale | Explicația plauzibilă |
|---|---|---|
| vestul francofon | Franța, Portugalia, Valonia și Bruxelles, Luxemburg, Elveția romandă | limba franceză; migrația portugheză spre Franța și Luxemburg |
| latin-mediteraneană | Italia, Spania, șase regiuni ale României, Ticino | migrația românească spre Italia și Spania |
| sud-estul și diaspora turcă | Turcia, Grecia, Bulgaria, Albania, vestul și nordul Germaniei, Țările de Jos | migrația turcă spre Germania și Țările de Jos |
| spațiul dunărean | sudul și estul Germaniei, Austria, Ungaria, fosta Iugoslavie, Elveția germană, Vestul și Centrul României | istoria comună (Austro-Ungaria), minoritățile germane și maghiare, migrația din fosta Iugoslavie |
| nordul și centrul-estul | Polonia, Cehia, Slovacia, țările nordice și baltice, Irlanda | migrația poloneză și baltică spre nord și spre Irlanda |

România se împarte în două: Vestul și Centrul (Banatul și Transilvania) intră în spațiul dunărean, cu Ungaria, Austria și Germania de sud; celelalte regiuni intră cu Italia și Spania, unde trăiesc cele mai mari comunități de români din Uniune.

In [ ]:
regiuni_ro = harta[harta["tara"] == "RO"]                 # regiunile României
regiuni_ro[["denumire", "comunitate"]]

Comunitățile spun *că* aceste regiuni sunt mai legate decât prezic distanța și granițele, nu *de ce*. Explicațiile din tabel sunt ipoteze: migrația se poate verifica cu datele din proiect, limba și istoria cu alte surse.

**Tipurile de regiuni și comunitățile.**
Două feluri de a grupa aceleași regiuni: după **ce sunt** (tipurile din secțiunea 8) și după **cu cine sunt legate** (secțiunea 9.4). Se leagă regiunile asemănătoare între ele, metropolele cu metropolele, periferiile cu periferiile? ARI răspunde direct.

In [ ]:
grupa_regiunii = reg.set_index("regiune")["grupa"]                   # regiunea -> tipul
comune = grupa_regiunii.index.intersection(comunitati.index)         # regiunile cu ambele etichete

adjusted_rand_score(grupa_regiunii[comune], comunitati[comune])

ARI este aproape 0: tipurile de regiuni și comunitățile nu au aproape nimic în comun. Regiunile nu se leagă după cât de bogate, tinere sau dense sunt, ci după migrație, limbă și istorie. Metropolele nu formează o comunitate a lor: București-Ilfov este legat de Italia și Spania, nu de Praga sau Madrid.

## 10. Ce se poate concluziona
- **Unitatea de analiză este perechea de țări.** SCI-ul țărilor (din C5), migrația și distanța sunt toate între țări.
- **Asociere, nu cauză.** Perechile cu mai mulți migranți sunt mai legate, la aceeași distanță și graniță; legăturile pot atrage și ele migrație.
- **Modelul final este M4.** Comparând partenerii aceleiași țări, contează distanța, numărul de migranți, granița, fostul stat comun și familia lingvistică, cu semnul așteptat; modelul greșește de obicei de 1,32 ori, față de 2,2 ori în M2.
- **Ce rămâne după model are explicații proprii.** Ungaria-România, Cehia-Slovacia, Luxemburg-Portugalia; pe regiuni, cinci comunități care trec peste granițe și urmează migrația, limba și istoria, nu asemănarea economică.
- **Perechile nu sunt independente.** Erorile grupate și regresia QAP ajung la aceeași concluzie pentru M2: fără efectele de țară, rezistă doar migrația.